## 3. Method 2: Positional Embedding of Peaks

### 3.1 Motivation

* Required Reading: https://www.nature.com/articles/s41467-024-49731-x
* Simple hashing and binning ignore relative *m/z* spacing.
* Many deep learning models require dense, continuous representations.

### 3.2 Encoding Strategy

* Apply **sinusoidal positional encoding** to each peak's *m/z* value.
* Each peak is mapped to a **512-dimensional vector**.
* Intensities can be incorporated as additional features or scaling factors.

### 3.3 Applications

* Captures fine-grained relationships between peaks.
* Useful as input to transformer or deep learning architectures for spectral analysis.

![Each m/z value, m_j, is projected into 512 dimensions](CasanovoEncoding.png)

Positional encoding, as outlined in the Casanovo paper. D = 512

In [ ]:
import numpy as np
from pyteomics import mzml
import spectrum_utils.plot as sup
import spectrum_utils.spectrum as sus
import pandas as pd
import matplotlib.pyplot as plt
from typing import Tuple, Dict

In [ ]:
def make_wavelengths(d_model: int, min_lambda: float, max_lambda: float) -> np.ndarray:
    """Create exponentially spaced wavelengths for use in sinusoidal encoding.
    
    Args:
        d_model (int): d_model/2 = 
        min_lambda (float): _description_
        max_lambda (float): _description_

    Returns:
        np.ndarray: _description_
    """
    

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
 
def make_wavelengths_casanovo(d_model: int, lambda_min: float = 0.001, lambda_max: float = 10000) -> np.ndarray:
    """
    Generate wavelengths following the Casanovo formula:
    λ_i = (λ_min / 2π) * (λ_max / λ_min)^(i / (d_sin - 1))
    
    Args:
        d_model = number of sin/cos pairs to consider.
        lambda_min: minimum wavelength (default: 0.001, as in Casanovo)
        lambda_max: maximum wavelength (default: 10000, as in Casanovo)
    
    Returns:cas
        Array of H wavelengths
    """
    # d_model must be divisible by 2.
    wavelengths = np.zeros(d_model // 2)
    for i in range(d_model//2):
        wavelengths[i] = (lambda_min / (2 * np.pi)) * (lambda_max / lambda_min) ** (i / (d_model//2 - 1))
    print("wavelengths:", wavelengths)
    return wavelengths


def plot_encoding_circle(
    mz_array,
    idx: int,
    lambdas=None,
    d_model: int = 512,
    lambda_min: float = 0.001,
    lambda_max: float = 10000,
    annotate: bool = True,
    show_formula: bool = False,
):
    """
    Plot the unit circle following the Casanovo positional encoding formula:
        sin(m_j / λ_i)  for i ≤ d/2
        cos(m_j / λ_i)  for i > d/2
    
    where λ_i = (λ_min / 2π) * (λ_max / λ_min)^(i / (d_sin - 1))

    Args:
      mz_array: 1-D array-like of m/z values
      idx:      which peak to visualize
      lambdas:  iterable of wavelengths to use (if given, H is ignored)
      H:        number of sin/cos pairs (if lambdas=None, default: 12)
      lambda_min: minimum wavelength (default: 0.001, as in Casanovo)
      lambda_max: maximum wavelength (default: 10000, as in Casanovo)
      annotate: label points with λ and dimension indices
      show_formula: display the formula on the plot
    """
    mz_array = np.asarray(mz_array, dtype=float).reshape(-1)
    if idx < 0 or idx >= mz_array.size:
        raise IndexError("idx out of bounds for mz_array.")
    m = float(mz_array[idx])

    lambdas = make_wavelengths_casanovo(d_model, lambda_min, lambda_max)
    d_model = len(lambdas)

    # Compute angles: m_j / λ_i
    angles = m / lambdas
    
    # First half: sin values (dimensions 0 to d/2)
    sin_vals = np.sin(angles)
    # Second half: cos values (dimensions d/2 to d)
    cos_vals = np.cos(angles)

    # Prepare unit circle
    theta = np.linspace(0, 2*np.pi, 400)
    circle_x = np.cos(theta)
    circle_y = np.sin(theta)

    # Create plot
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.plot(circle_x, circle_y, 'k-', linewidth=1.5, alpha=0.3, label='Unit circle')
    
    # Plot sin coordinates (first half of dimensions)
    ax.scatter(cos_vals, sin_vals, s=80, c='blue', marker='o', 
               alpha=0.7, edgecolors='darkblue', linewidth=1.5,
               label=f'sin(m/λ) [dims 0-{d_model//2-1}]', zorder=5)

    # Cross-hairs
    ax.axhline(0, color='gray', linewidth=0.8, linestyle='--', alpha=0.5)
    ax.axvline(0, color='gray', linewidth=0.8, linestyle='--', alpha=0.5)

    if annotate:
        for i, (c, s, lam) in enumerate(zip(cos_vals, sin_vals, lambdas)):
            # Annotate with dimension index and wavelength
            ax.annotate(f'd={i}\nλ={lam:.2e}', 
                       (c, s), 
                       textcoords="offset points", 
                       xytext=(8, 8),
                       fontsize=8,
                       bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.5))

    ax.set_aspect('equal', adjustable='box')
    ax.set_xlim(-1.15, 1.15)
    ax.set_ylim(-1.15, 1.15)
    ax.set_xlabel('cos(m/λ)', fontsize=12, fontweight='bold')
    ax.set_ylabel('sin(m/λ)', fontsize=12, fontweight='bold')
    ax.set_title(f'Casanovo Positional Encoding Circle\nm/z = {m:.4f} (peak index {idx})', 
                fontsize=13, fontweight='bold', pad=15)
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(True, alpha=0.2)
    
    if show_formula:
        formula_text = (
            r'$f_i(m_j) = \begin{cases}'
            r'\sin(m_j / \lambda_i) & \text{for } i \leq d/2 \\'
            r'\cos(m_j / \lambda_i) & \text{for } i > d/2'
            r'\end{cases}$' + '\n' +
            r'$\lambda_i = \frac{\lambda_{min}}{2\pi} \left(\frac{\lambda_{max}}{\lambda_{min}}\right)^{i/(d_{sin}-1)}$'
        )
        ax.text(0.02, 0.98, formula_text, transform=ax.transAxes,
               fontsize=9, verticalalignment='top',
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    plt.tight_layout()
    plt.show()

# Example: Plot encoding circles for just the first 3 peaks of the first spectrum
# (to avoid generating too many plots)
ls = [i/100 for i in range(10000,11000,1)]
for i in range(len(ls)):
    plot_encoding_circle(ls, i, d_model = 512, annotate=True)


In [ ]:
# Define positional encoding function used by Casanovo
# This function takes in 1 m/z value and returns a 512-dimensional positional encoding vector
# (this feels sort of like a reverse Fourier transform)
def positional_encoding(m_z, d_model = 512, lambda_min = 0.001, lambda_max = 10000):
    encoding = np.zeros(d_model)
    d_sin = int(d_model/2)
    d_cos = d_model - d_sin
    for d in range(d_sin):
        encoding[d] = np.sin(m_z / ( (lambda_min / (2 * np.pi) ) * (lambda_max / lambda_min)**(d/(d_sin - 1))))
    for d in range(d_sin, d_model):
        encoding[d] = np.cos(m_z / ( (lambda_min / (2 * np.pi) ) * (lambda_max / lambda_min)**((d - d_sin ) / (d_cos - 1))))
    return encoding


In [ ]:
encode_1 = positional_encoding(1000)
encode_2 = positional_encoding(1000.1)
encode_3 = positional_encoding(126.1277)
for i in range(50,2000):
    positional_encoding(i)
display(np.dot(positional_encoding(1000),(positional_encoding(1000.1))), "and", np.dot(positional_encoding(1000),positional_encoding(126.127)))

# plot the encodings as three scatter plots,

# Set the figure size before creating subplots
plt.figure(figsize=(12, 10))

plt.subplot(3, 1, 1)
plt.scatter(range(len(encode_1)), encode_1)
plt.title("Positional Encoding - 1000 m/z")
plt.xlabel("Dimension (i)")
plt.ylabel("Value")

plt.subplot(3, 1, 2)
plt.scatter(range(len(encode_2)), encode_2)
plt.title("Positional Encoding - 1000.1 m/z")
plt.xlabel("Dimension (i)")
plt.ylabel("Value")

plt.subplot(3, 1, 3)
plt.scatter(range(len(encode_3)), encode_3)
plt.title("Positional Encoding - 126.128 m/z")
plt.xlabel("Dimension (i)")
plt.ylabel("Value")

plt.tight_layout()
plt.show()


I only halfway understand what's going on here, so the following explanation should be taken with a grain of salt

The positional encoding attempts to capture both high-resolution information and low-resolution information.

Dimensions ~200-255 and ~450-500 capture low resolution information. They will only differ if there is a large m/z difference between the encoded values.

Dimensions 0-50 and 256-300 encoded high resolution information. They change even if the m/z values are very close together

In [ ]:
# for encode_1 and encode_2, plot the values from 200 to 255
# and calculate the cosine similarity between the two
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(range(200, 256), encode_1[200:256])
plt.title("Positional Encoding - 1000 m/z (200-255)")
plt.xlabel("Dimension (i)")
plt.ylabel("Value")

plt.subplot(1, 2, 2)
plt.scatter(range(200, 256), encode_2[200:256])
plt.title("Positional Encoding - 1000.1 m/z (200-255)")
plt.xlabel("Dimension (i)")
plt.ylabel("Value")

plt.tight_layout()
plt.show()
from sklearn.metrics.pairwise import cosine_similarity

# Calculate cosine similarity between the two encodings
cosine_sim = cosine_similarity([encode_1[200:256]], [encode_2[200:256]])
print("Cosine Similarity (1000 m/z vs 1000.1 m/z):", cosine_sim[0][0])

In [ ]:
# Repeat the above plots and calculations, but this time only consider the range from 0 to 56
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(range(0, 57), encode_1[0:57])
plt.title("Positional Encoding - 1000 m/z (0-56)")
plt.xlabel("Dimension (i)")
plt.ylabel("Value")

plt.subplot(1, 2, 2)
plt.scatter(range(0, 57), encode_2[0:57])
plt.title("Positional Encoding - 1000.1 m/z (0-56)")
plt.xlabel("Dimension (i)")
plt.ylabel("Value")

plt.tight_layout()
plt.show()

# Calculate cosine similarity between the two encodings
cosine_sim = cosine_similarity([encode_1[0:57]], [encode_2[0:57]])
print("Cosine Similarity (1000 m/z vs 1000.1 m/z):", cosine_sim[0][0])